# Climatology calculations

In [ ]:
from earthkit.data.testing import earthkit_remote_test_data_file

from earthkit import data as ekd
from earthkit import plots as ekp
from earthkit import transforms as ekt

## Load test data

In this example we will use hourly ERA5 2m temperature data on a 3˚ X 3˚ spatial grid for the years 2015 to 2017 as
our physical data.

All `earthkit-transforms` methods can be called with `earthkit-data` objects (Readers and Wrappers)
or with a pre-loaded `xarray`.
To reduce the number of conversions in the example, we will convert to xarray in the first cell
and use that data object for all subsequent steps. This also allows us to set some dataset specific options
in the conversion to xarray instead of relying on the default options.

In [ ]:
# Get some demonstration ERA5 data, this could be any url or path to an ERA5 grib or netCDF file.
remote_era5_file = earthkit_remote_test_data_file("era5-Europe-sfc-2m-temperature-3deg-2015-2017.grib")
era5_data = ekd.from_source("url", remote_era5_file)

# convert to xarray to save repeated conversion in further steps.
# the .compute loads everything into memory which is safe for this example
era5_xr = era5_data.to_xarray(time_dim_mode="valid_time").compute()
era5_xr

## Calculate the climatologies

### Monthly mean

First we calculate the monthly mean. In the returned object the valid_time dimension has been replaced with a
`month` dimension with values `1` to `12` corresponding to months January to December.

In [ ]:
climatology_monthly_mean = ekt.climatology.monthly_mean(era5_xr)
climatology_monthly_mean

### Daily mean, min, max and standard deviation

Next we repeat for the daily mean, this time the `valid_time` is replaced with a `dayofyear` dimension with values 1 to 366.

The 366th value is due to leap years, and it is the 366th value that is under-sampled (i.e. one value every 4 years).

In [ ]:
climatology_daily_mean = ekt.climatology.daily_mean(era5_xr)
climatology_daily_mean

Repeat for the daily maximum, minimum  and standard deviation

In [ ]:
climatology_daily_max = ekt.climatology.daily_max(era5_xr)
climatology_daily_min = ekt.climatology.daily_min(era5_xr)
climatology_daily_std = ekt.climatology.daily_std(era5_xr)
climatology_daily_std

### Monthly quantiles

Please note the api for quantiles is slightly different, it requires an additional argument `q` which is a
list of the quantiles to return.

Additionally, the returned object has `quantiles` dimension which is for each of the quantiles returned.

In [ ]:
climatology_monthly_quantiles = ekt.climatology.quantiles(era5_xr, [0.1, 0.5, 0.9], frequency="month")
climatology_monthly_quantiles

## Plot the output for a random location

First we plot the climatology data using an earthkit-plots `Climatology` figure.

In [ ]:
latitude = 51.5
longitude = -1.0
sel_kwargs = {"latitude": latitude, "longitude": longitude, "method": "nearest"}

chart = ekp.Climatology(wrap_time=True)

chart.fix_y_units("celsius")

# Monthly mean + q10/q90 envelope
chart.line(
    climatology_monthly_mean.sel(**sel_kwargs), label="Monthly mean", color="seagreen", drawstyle="spline"
)
chart.fill_between(
    climatology_monthly_quantiles.sel(**sel_kwargs, quantile=0.1),
    climatology_monthly_quantiles.sel(**sel_kwargs, quantile=0.9),
    label="Monthly $Q_{10} / Q_{90}$ range",
    color="seagreen",
    drawstyle="spline",
)

# Daily mean + min/max envelope
chart.line(
    climatology_daily_mean.sel(**sel_kwargs),
    label="Daily mean",
    color="cornflowerblue",
)
chart.fill_between(
    climatology_daily_min.sel(**sel_kwargs),
    climatology_daily_max.sel(**sel_kwargs),
    label="Daily min/max range",
    color="cornflowerblue",
)

chart.title(
    "Daily and monthly climatology for {variable_name}\n"
    "{location:%c}, {location:%C} ({latitude:%Lt}, {longitude:%Ln})"
)
chart.xticks(frequency="M", period=True)
chart.ylabel()
chart.legend()
chart.show()

Now we add the climatology to the background of the Timeseries plot for the daily mean on the original `valid_time` dimension.
First we create the repeated climatology for the monthly quantiles, then we plot with `earthkit-plots`

In [ ]:
latitude = 51.5
longitude = -1.0
sel_kwargs = {"latitude": latitude, "longitude": longitude, "method": "nearest"}

chart = ekp.TimeSeries()

# repeat_years tiles the month-dimensioned climatology across 2015–2017
# automatically — no dummy datetime coordinates needed.
chart.fill_between(
    climatology_monthly_quantiles.sel(**sel_kwargs, quantile=0.1),
    climatology_monthly_quantiles.sel(**sel_kwargs, quantile=0.9),
    repeat_years=range(2015, 2018),
    units="celsius",
    label="Monthly climatology ($Q_{10} / Q_{90}$ range)",
    color="seagreen",
    drawstyle="spline",
)

era5_daily_mean = ekt.temporal.daily_mean(era5_xr)
chart.line(
    era5_daily_mean.sel(**sel_kwargs), units="celsius", label="Daily mean ERA5 data", color="firebrick", lw=1
)

chart.title(
    "Raw hourly data and repeated monthly climatology for {variable_name}\n"
    "{location:%c}, {location:%C} ({latitude:%Lt}, {longitude:%Ln})"
)

chart.xticks(frequency="6M", format="%B %Y")

chart.ylabel()
chart.legend()
chart.show()